In [1]:
import napari
import skimage as ski
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
import glob
import dask
import cv2
import scipy.ndimage as ndi
import pandas as pd
import plotly.express as px

In [2]:
viewer = napari.Viewer()

In [3]:
fnames = glob.glob('data/*mask.tif')
fnames

['data\\ncol5_488_ncol1_546_dapi_ncol4_647_s01_mask.tif',
 'data\\ncol5_488_ncol1_546_dapi_ncol4_647_s02_mask.tif',
 'data\\ncol5_488_ncol1_546_dapi_ncol4_647_s03_mask.tif',
 'data\\ncol5_488_ncol1_546_dapi_ncol4_647_s04_mask.tif',
 'data\\ncol5_488_ncol1_546_dapi_ncol4_647_s06_mask.tif',
 'data\\ncol5_488_ncol1_546_dapi_ncol4_647_s10_mask.tif']

In [4]:
fnames = ['data\\ncol5_488_ncol1_546_dapi_ncol4_647_s01_mask.tif',
          'data\\ncol5_488_ncol1_546_dapi_ncol4_647_s03_mask.tif', 
          'data\\ncol5_488_ncol1_546_dapi_ncol4_647_s02_mask.tif',
          'data\\ncol5_488_ncol1_546_dapi_ncol4_647_s04_mask.tif',
          'data\\ncol5_488_ncol1_546_dapi_ncol4_647_s06_mask.tif'  ]

In [5]:
idx = 4
fname = fnames[idx]

In [6]:
img = ski.io.imread(fnames[idx].replace('_mask',''))[1:]

In [7]:
viewer.add_image(img, channel_axis=0, scale=(10,1,1))

[<Image layer 'Image' at 0x1bb3338e5d0>,
 <Image layer 'Image [1]' at 0x1bb3338cdd0>,
 <Image layer 'Image [2]' at 0x1bb2d8ffe30>,
 <Image layer 'Image [3]' at 0x1bb2d9a3c20>]

In [8]:
mask = ski.io.imread(fname)

In [9]:
viewer.add_labels(mask*3, name='mask', scale=(10,1,1))

<Labels layer 'mask' at 0x1bb192342c0>

In [10]:
ds_mask = ski.transform.downscale_local_mean(mask, (1, 4, 4), cval=0)
ds_mask[:,0,:] = 0
ds_mask[:,-1,:] = 0
ds_mask[:, :, 0] = 0
ds_mask[:, :, -1] = 0

In [11]:
edt = ndi.distance_transform_edt(ds_mask, sampling=(10,4,4))
viewer.add_image(edt, name='edt', scale=(10,4,4))

<Image layer 'edt' at 0x1bc5840dee0>

In [12]:
us_edt = ski.transform.resize(edt, (img.shape[1], img.shape[2], img.shape[3]), order=1, )

In [13]:
z = np.arange(0, img.shape[1])
z_scaler = np.exp(-z/100)
clipped = np.clip(img.astype(np.float32)-95.0,0.0, np.inf)
z_corrected = clipped/z_scaler[np.newaxis, :, np.newaxis, np.newaxis]

In [14]:
viewer.add_image(z_corrected, name='z corrected', contrast_limits=[0,1138], scale=(10,1,1), channel_axis=0)

[<Image layer 'z corrected' at 0x1bc5839c2c0>,
 <Image layer 'z corrected [1]' at 0x1bcfe7cfef0>,
 <Image layer 'z corrected [2]' at 0x1bc584d40e0>,
 <Image layer 'z corrected [3]' at 0x1bcfe7cd310>]

In [15]:
viewer.add_image(us_edt, name='us_edt', scale=(10,1,1),)

<Image layer 'us_edt' at 0x1bcfe7cda90>

In [16]:
shell_cutoff = 60

In [17]:
filtered = z_corrected * (us_edt>shell_cutoff)[np.newaxis,:,:,:]
viewer.add_image(filtered, name='filtered', contrast_limits=[0,1138], scale=(10,1,1), channel_axis=0)

[<Image layer 'filtered' at 0x1bb26849cd0>,
 <Image layer 'filtered [1]' at 0x1bb26821970>,
 <Image layer 'filtered [2]' at 0x1bb26de7260>,
 <Image layer 'filtered [3]' at 0x1bb26e1cb60>]

# Try Pearsons instead

In [18]:
def gaussian_filter(inp, sz=17):
    rtn = cv2.GaussianBlur(inp, (sz, sz), 0)
    return rtn

In [19]:
def blur_channel(timg, sz=11):
    all_data = []
    delayed = dask.delayed(gaussian_filter)
    for i, f in enumerate(timg):
        all_data.append(delayed(f, sz))
    all_data = dask.compute(*all_data)  
    # for i, f in enumerate(timg):
    #     all_data.append(gaussian_filter(f))
    all_data = np.array(all_data)
    return all_data


In [20]:
def blur(filtered, sz=11):
    blurred = filtered.astype(np.uint16).copy()
    for idx in range(0,(filtered.shape[0])):
        print(idx)
        blurred[idx,:,:,:] = blur_channel(filtered[idx,:,:,:],sz)
    return blurred

In [21]:
def filter_size(mask, min_size=100):
    labels = ski.measure.label(mask, connectivity=1)
    props = ski.measure.regionprops(labels)
    areas = np.array([prop.area for prop in props])
    labs = np.array([prop.label for prop in props])
    filtered = ski.util.map_array(labels, labs, (areas > min_size) * labs)
    return filtered

In [22]:
animal_mask = us_edt > shell_cutoff

In [23]:
short_animal_mask = animal_mask[10:]

In [24]:
def get_pearsons(blurred, c0, c1, c3, a_otsu, b_otsu, c_otsu, min_size=200, otsu_based=True):
    A = blurred[c0,10:,:,:].astype(np.float32)
    B = blurred[c1,10:,:,:].astype(np.float32)
    C = blurred[c3,10:,:,:].astype(np.float32)

    A = A - np.mean(A[short_animal_mask])
    B = B - np.mean(B[short_animal_mask])
    C = C - np.mean(C[short_animal_mask])
    A = A / np.std(A[short_animal_mask])
    B = B / np.std(B[short_animal_mask])
    C = C / np.std(C[short_animal_mask])

    A_thresh = a_otsu * ski.filters.threshold_otsu(A[short_animal_mask])
    B_thresh = b_otsu * ski.filters.threshold_otsu(B[short_animal_mask])
    C_thresh = c_otsu * ski.filters.threshold_otsu(C[short_animal_mask])
    
    if otsu_based == False:
        A_thresh = a_otsu
        B_thresh = b_otsu
        C_thresh = c_otsu
    print(A_thresh, B_thresh, C_thresh)

    A_mask = A > A_thresh
    B_mask = B > B_thresh
    C_mask = C > C_thresh

    A_label = ndi.label(A_mask)[0]
    B_label = ndi.label(B_mask)[0]
    C_label = ndi.label(C_mask)[0]

    A_label = filter_size(A_label, min_size=min_size)
    B_label = filter_size(B_label, min_size=min_size)
    C_label = filter_size(C_label, min_size=min_size)

    A_label = A_label > 0
    B_label = B_label > 0
    C_label = C_label > 0

    A_B_pearson = ski.measure.pearson_corr_coeff((A_label).astype(np.float32), (B_label).astype(np.float32), mask=short_animal_mask)
    A_C_pearson = ski.measure.pearson_corr_coeff((A_label).astype(np.float32), (C_label).astype(np.float32), mask=short_animal_mask)
    B_C_pearson = ski.measure.pearson_corr_coeff((B_label).astype(np.float32), (C_label).astype(np.float32), mask=short_animal_mask)
    return [A_B_pearson[0], A_C_pearson[0], B_C_pearson[0]], [A_label, B_label, C_label], [A, B, C]

In [25]:
blurred = blur(filtered, sz=11)

0
1
2
3


In [26]:
thresholds = [[0.8,2.2,0.8], [-30.0,1.8,0.8], [10.0,1.5,1.5],[5.0,3.2,4.3], [5.0,3.2,2.5]]
otsus = [True, True, False, False, False]
c0_otsu, c1_otsu, c3_otsu = thresholds[idx]

# # These work for idx = 0, s01
# c0_otsu = 1.6
# c1_otsu = 1.8
# c3_otsu = 1.0

# # These work for idx = 1, s01
# c0_otsu = -30.0
# c1_otsu = 1.8
# c3_otsu = 0.8

pearsons,labels,normed = get_pearsons(blurred, 0, 1, 3, c0_otsu, c1_otsu, c3_otsu, min_size=400, otsu_based=otsus[idx])
c0,c1,c3 = labels
c0normed,c1normed,c3normed = normed
print(pearsons)

5.0 3.2 2.5
[0.01738899201154709, 0.014769749715924263, 0.5042752623558044]


In [27]:
viewer.add_image(c0normed, blending='additive', colormap='magenta')
viewer.add_image(c1normed, blending='additive', colormap='green')
viewer.add_image(c3normed, blending='additive', colormap='magenta')

viewer.add_image(c0, blending='additive', colormap='magenta', opacity=0.5)
viewer.add_image(c1, blending='additive', colormap='green', opacity=0.5)
viewer.add_image(c3, blending='additive', colormap='magenta', opacity=0.5)

<Image layer 'c3' at 0x1bd006ccfb0>

In [28]:
np.sum(c0*c3)/np.sum(c0)

np.float64(0.06235016888312471)

In [38]:
# [0.053803956797661576, 0.08869494231776619, 0.44717610574280864]
# [0.017856292077720666, 0.033245664466067526, 0.44414741782992045]
# [0.02473398690140375, 0.027990753599469672, 0.5441216220269873]
# [0.042567058781666514, 0.0246119752652963, 0.47137659090156836]
# [0.017386938961034867, 0.014766023186774897, 0.5043098240185087]

# Plot results

In [29]:
data = np.array([[0.053803956797661576, 0.08869494231776619, 0.44717610574280864],
[0.017856292077720666, 0.033245664466067526, 0.44414741782992045],
[0.02473398690140375, 0.027990753599469672, 0.5441216220269873],
[0.042567058781666514, 0.0246119752652963, 0.47137659090156836],
[0.017386938961034867, 0.014766023186774897, 0.5043098240185087]])

In [30]:
df = pd.DataFrame(data, columns=['ncol5-ncol1', 'ncol5-ncol4', 'ncol1-ncol4'], index=['s01', 's03', 's02', 's04', 's06'])
melted = df.reset_index().melt(id_vars='index')
melted.columns = ['Sample', 'Pearson', 'Value']

In [31]:
f = px.box(melted, x='Pearson', y='Value', points='all', title='Pearson Correlation Coefficient', width=500)
f.write_html('pearson.html')
f.write_image('pearson.png', scale=2)
f.write_image('pearson.pdf', scale=2)  # Save as PDF for high-quality printout
f